# CV教材：画像を種類に分ける

## はじめに

このNotebookでは、画像分類の小さな課題を作り、同じ条件で結果を確かめます。
使うデータは、出典と固定版を確認したトマトの葉の画像です。画像の判定は植物の病気の確定診断ではありません。

## 1. 自分の課題を決める

画像から何を分けたいか、誰の役に立つか、成功したといえる条件を書きます。

- 困りごと：
- 分けたい種類：
- 使う人：
- 成功したといえる条件：
- 試す入力と期待する出力：

## 2. データの出典と利用条件を確認する

このNotebookでは、次の固定版だけを使います。

- データ：トマトの葉 5クラス
- 取得ID：`tomato_leaves` または `トマトの葉`
- 固定版：`tomato-v1.2.0`
- 元データの表記：CC0 1.0
- 注意：画像の判定は医療・農業上の確定診断ではありません。

元画像をNotebookへ再収録したり、ラベルを書き換えたりしません。

In [ ]:
# 先生が指定したCV_dataのフォルダで、最初に一度だけ実行します。
# %pip install -e '.[dataframe,display,torch]'

from cvlesson import datasets
import pandas as pd

data = datasets.get('トマトの葉')
df = data.to_df()
print(data.source_info)
print('クラス:', data.class_names_ja)
print('分割:', data.splits)
df.head()

## 3. 画像とラベルを確認する

`label_ja`は日本語で表示するための名前、`label_id`は学習で使う番号です。
`test`は最後の評価用なので、学習には使いません。

In [ ]:
print(df[['image_path', 'label', 'label_ja', 'label_id', 'split', 'sample_id']].head())
print(df['label_ja'].value_counts())
data.show_examples(9, split='train')

## 4. 小さな分類モデルを動かす

画像は必要なときに読み込みます。学習用・検証用・テスト用の画像は、`cvlesson`が作った同じ分割を使います。

まずはCPUで動かせる小さなモデルを試します。正解率だけでなく、どの種類を間違えたかも確認します。

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset

画像サイズ = (64, 64)
train_all = data.to_torch_dataset(split='train', image_size=画像サイズ)
val_all = data.to_torch_dataset(split='val', image_size=画像サイズ)
test_data = data.to_torch_dataset(split='test', image_size=画像サイズ)

def クラスごとに選ぶ(dataset, 枚数=100, seed=42):
    generator = torch.Generator().manual_seed(seed)
    selected = []
    for label_id in range(len(data.classes)):
        positions = [i for i, value in enumerate(dataset.targets) if int(value) == label_id]
        order = torch.tensor(positions)[torch.randperm(len(positions), generator=generator)]
        selected.extend(order[:枚数].tolist())
    return Subset(dataset, selected)

train_data = クラスごとに選ぶ(train_all, 100)
val_data = クラスごとに選ぶ(val_all, 30)

class SmallCNN(nn.Module):
    def __init__(self, classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)))
        self.classifier = nn.Linear(16 * 4 * 4, classes)
    def forward(self, x):
        return self.classifier(self.features(x).flatten(1))

model = SmallCNN(len(data.classes))
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
loss_fn = nn.CrossEntropyLoss()
for epoch in range(2):
    model.train()
    for images, labels in DataLoader(train_data, batch_size=32, shuffle=True):
        optimizer.zero_grad()
        loss = loss_fn(model(images), labels)
        loss.backward()
        optimizer.step()
    model.eval()
    correct = sum((model(x).argmax(1) == y).sum().item() for x, y in DataLoader(val_data, batch_size=64))
    print(f'{epoch + 1}回目：検証用正解率 {correct / len(val_data):.1%}')

## 5. 見本を試す

テスト用画像に対する予測を表示します。`sample:0`から`sample:4`までを試し、正解ラベルと比べます。

In [ ]:
import matplotlib.pyplot as plt

def 予測する(index):
    image, label = test_data[index]
    with torch.no_grad():
        probability = torch.softmax(model(image.unsqueeze(0)), dim=1)[0]
    predicted = int(probability.argmax())
    plt.imshow(image.permute(1, 2, 0))
    plt.axis('off')
    plt.title(f'予測: {data.class_names_ja[predicted]} / 正解: {data.class_names_ja[int(label)]}')
    plt.show()
    return {'予測': data.class_names_ja[predicted], '正解': data.class_names_ja[int(label)], '確率': float(probability[predicted])}

for index in range(5):
    print(index, 予測する(index))

## 6. 自分の課題に合わせて試す

次の欄に、試した番号、期待した結果、実際の結果を書きます。モデルが間違えたときは、画像の見え方や種類の似ている点も記録します。

- 試した入力：
- 期待した結果：
- 実際の結果：
- 次に変えたいこと：

In [ ]:
# 自分で試すテスト番号を変更します。testの画像だけを使います。
試す番号 = 0
予測する(試す番号)

## 7. 振り返り

- うまく分類できた種類：
- 間違えやすかった種類：
- test画像で確かめたこと：
- データの制約や、まだ分からないこと：
- 次に改善するなら：